# Website Automation: DANA Movilidad

This notebook automates interaction with the FME Server application at: https://fme.conterra.de/fmeserver/apps/DANAMovilidad

The automation will:
1. Set a sample date
2. Select Excel format
3. Click the Run button

In [ ]:
# Import required libraries
import time
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException

print("Libraries imported successfully!")

In [ ]:
# Setup Chrome driver with options
def setup_driver():
    chrome_options = Options()
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--window-size=1920,1080')
    
    # For headless operation, uncomment the line below
    # chrome_options.add_argument('--headless')
    
    try:
        driver = webdriver.Chrome(options=chrome_options)
        print("Chrome driver initialized successfully")
        return driver
    except Exception as e:
        print(f"Error initializing Chrome driver: {e}")
        return None

# Initialize the driver
driver = setup_driver()

In [ ]:
# Navigate to the website
url = "https://fme.conterra.de/fmeserver/apps/DANAMovilidad"

if driver:
    try:
        print(f"Navigating to: {url}")
        driver.get(url)
        
        # Wait for the page to load
        wait = WebDriverWait(driver, 10)
        print("Page loaded successfully")
        
        # Give some time for dynamic content to load
        time.sleep(3)
        
    except Exception as e:
        print(f"Error navigating to website: {e}")
else:
    print("Driver not initialized. Cannot proceed.")

In [ ]:
# Function to find and set date input
def set_sample_date(driver, wait):
    try:
        # Sample date - let's use a recent date
        sample_date = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')
        print(f"Setting sample date: {sample_date}")
        
        # Common selectors for date inputs
        date_selectors = [
            'input[type="date"]',
            'input[placeholder*="date"]',
            'input[name*="date"]',
            'input[id*="date"]',
            '.date-picker input',
            '[data-type="date"]'
        ]
        
        date_input = None
        for selector in date_selectors:
            try:
                date_input = driver.find_element(By.CSS_SELECTOR, selector)
                print(f"Found date input with selector: {selector}")
                break
            except NoSuchElementException:
                continue
        
        if date_input:
            # Clear and set the date
            date_input.clear()
            date_input.send_keys(sample_date)
            print(f"Date set to: {sample_date}")
            return True
        else:
            print("No date input found with common selectors")
            return False
            
    except Exception as e:
        print(f"Error setting date: {e}")
        return False

# Set the sample date
if driver:
    wait = WebDriverWait(driver, 10)
    date_set = set_sample_date(driver, wait)
else:
    print("Driver not available")

In [ ]:
# Function to set Excel format
def set_excel_format(driver, wait):
    try:
        print("Looking for Excel format option...")
        
        # Common selectors for format selection
        format_selectors = [
            'select[name*="format"]',
            'select[id*="format"]',
            'select[name*="output"]',
            'select[id*="output"]',
            '.format-selector select',
            '.output-format select',
            'select option[value*="excel"]/../..',
            'select option[value*="xlsx"]/../..'
        ]
        
        format_dropdown = None
        for selector in format_selectors:
            try:
                format_dropdown = driver.find_element(By.CSS_SELECTOR, selector)
                print(f"Found format dropdown with selector: {selector}")
                break
            except NoSuchElementException:
                continue
        
        if format_dropdown:
            select = Select(format_dropdown)
            
            # Try to find Excel option by various values
            excel_options = ['excel', 'xlsx', 'xls', 'Excel', 'EXCEL', 'Microsoft Excel']
            
            for option_value in excel_options:
                try:
                    select.select_by_value(option_value)
                    print(f"Selected Excel format with value: {option_value}")
                    return True
                except:
                    try:
                        select.select_by_visible_text(option_value)
                        print(f"Selected Excel format with text: {option_value}")
                        return True
                    except:
                        continue
            
            # If exact match not found, try partial match
            options = select.options
            for option in options:
                if any(excel_term.lower() in option.text.lower() for excel_term in ['excel', 'xlsx', 'xls']):
                    select.select_by_visible_text(option.text)
                    print(f"Selected Excel format: {option.text}")
                    return True
            
            print("No Excel format option found in dropdown")
            return False
        else:
            print("No format dropdown found")
            return False
            
    except Exception as e:
        print(f"Error setting Excel format: {e}")
        return False

# Set Excel format
if driver:
    excel_set = set_excel_format(driver, wait)
else:
    print("Driver not available")

In [ ]:
# Function to click the Run button
def click_run_button(driver, wait):
    try:
        print("Looking for Run button...")
        
        # Common selectors for run buttons
        run_selectors = [
            'button[type="submit"]',
            'input[type="submit"]',
            'button:contains("Run")',
            'input[value="Run"]',
            'button[id*="run"]',
            'button[name*="run"]',
            'button[class*="run"]',
            '.run-button',
            '.submit-button',
            'button[id*="submit"]',
            'button[name*="submit"]'
        ]
        
        run_button = None
        for selector in run_selectors:
            try:
                run_button = driver.find_element(By.CSS_SELECTOR, selector)
                print(f"Found run button with selector: {selector}")
                break
            except NoSuchElementException:
                continue
        
        # If CSS selectors don't work, try XPath
        if not run_button:
            xpath_selectors = [
                '//button[contains(text(), "Run")]',
                '//input[@value="Run"]',
                '//button[contains(text(), "Submit")]',
                '//button[contains(text(), "Execute")]',
                '//button[contains(text(), "Start")]',
                '//input[contains(@value, "Run")]'
            ]
            
            for xpath in xpath_selectors:
                try:
                    run_button = driver.find_element(By.XPATH, xpath)
                    print(f"Found run button with XPath: {xpath}")
                    break
                except NoSuchElementException:
                    continue
        
        if run_button:
            # Scroll to button if needed
            driver.execute_script("arguments[0].scrollIntoView();", run_button)
            time.sleep(1)
            
            # Click the button
            run_button.click()
            print("Run button clicked successfully!")
            return True
        else:
            print("No Run button found")
            return False
            
    except Exception as e:
        print(f"Error clicking Run button: {e}")
        return False

# Click the Run button
if driver:
    run_clicked = click_run_button(driver, wait)
else:
    print("Driver not available")

In [ ]:
# Alternative approach: Manual element inspection
# Use this cell if the automated approach doesn't work

if driver:
    print("=== MANUAL INSPECTION MODE ===")
    print("Current page source (first 500 characters):")
    print(driver.page_source[:500] + "...")
    
    print("\n=== ALL INPUT ELEMENTS ===")
    inputs = driver.find_elements(By.TAG_NAME, "input")
    for i, inp in enumerate(inputs[:10]):  # Show first 10 inputs
        print(f"Input {i+1}: type={inp.get_attribute('type')}, name={inp.get_attribute('name')}, id={inp.get_attribute('id')}")
    
    print("\n=== ALL SELECT ELEMENTS ===")
    selects = driver.find_elements(By.TAG_NAME, "select")
    for i, sel in enumerate(selects):
        print(f"Select {i+1}: name={sel.get_attribute('name')}, id={sel.get_attribute('id')}")
        options = sel.find_elements(By.TAG_NAME, "option")
        for opt in options[:5]:  # Show first 5 options
            print(f"  Option: {opt.text} (value={opt.get_attribute('value')})")
    
    print("\n=== ALL BUTTON ELEMENTS ===")
    buttons = driver.find_elements(By.TAG_NAME, "button")
    for i, btn in enumerate(buttons):
        print(f"Button {i+1}: text='{btn.text}', id={btn.get_attribute('id')}, class={btn.get_attribute('class')}")

In [ ]:
# Cleanup: Close the browser
if driver:
    print("Closing browser...")
    driver.quit()
    print("Browser closed successfully")
else:
    print("No browser to close")

## Instructions for Running

1. **Install Dependencies**: First, install the required packages:
   ```bash
   pip install selenium
   ```

2. **Install ChromeDriver**: Download ChromeDriver from https://chromedriver.chromium.org/ and ensure it's in your PATH, or install via:
   ```bash
   # On Ubuntu/Debian
   sudo apt-get install chromium-chromedriver
   
   # Or using webdriver-manager
   pip install webdriver-manager
   ```

3. **Run the Cells**: Execute the cells in order. The automation will:
   - Open a Chrome browser
   - Navigate to the DANA Movilidad website
   - Attempt to find and set a sample date
   - Try to select Excel as the output format
   - Click the Run button

4. **Troubleshooting**: If the automated approach doesn't work, use the "Manual Inspection" cell to see all available form elements and adjust the selectors accordingly.

5. **Customization**: You can modify the `sample_date` in the date setting function to use a specific date of your choice.

**Note**: The success of this automation depends on the website's structure. If the website uses dynamic loading or has specific security measures, you may need to adjust the selectors or add additional wait conditions.